In [10]:
import requests
import time
import random
import hashlib
import smtplib
import threading
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- CONFIGURATION ---

URL_TO_CHECK = "https://www.marathondumedoc.com/inscription/"

# --- LISTES DE DIFFUSION ---
LISTE_VIP = ["petillion99@gmail.com"]
LISTE_STANDARD = ["quentinlevdso@gmail.com"]

# --- CONFIGURATION EMAIL ---
EMAIL_SENDER = "pierre.elipse@gmail.com"
EMAIL_PASSWORD = "mifz iexy csrq xsjr"

# --- CONFIGURATION DISCORD ---
ENABLE_DISCORD = True
# Collez ici l'URL Webhook copiée depuis les paramètres de votre salon Discord
DISCORD_WEBHOOK_URL = "https://discordapp.com/api/webhooks/1480063682737078325/GnUT9XCJfqnd04moCAQO3h6xkGE5MAyTlhHB267JQ9m8uGnueBCNuqwQ7N7TM-2j0xYb"

MIN_INTERVAL = 50
MAX_INTERVAL = 70

# --- FONCTIONS ---

def get_content_hash(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            content = response.text
            return hashlib.md5(content.encode('utf-8')).hexdigest()
        else:
            print(f"Erreur HTTP : {response.status_code}")
            return None
    except Exception as e:
        print(f"Erreur lors de la récupération de la page : {e}")
        return None

def send_discord(text):
    """Envoie un message via Discord Webhook."""
    if not ENABLE_DISCORD:
        return
    try:
        # On prépare les données pour Discord
        data = {
            "content": text,
            # Vous pouvez changer le nom du bot qui envoie le message
            "username": "Marathon Bot"
        }

        response = requests.post(DISCORD_WEBHOOK_URL, json=data)

        if response.status_code == 204:
            print("[DISCORD] SUCCÈS.")
        else:
            print(f"[DISCORD] ERREUR : {response.text}")
    except Exception as e:
        print(f"[DISCORD] Erreur : {e}")

def send_email_to_list(recipients_list, subject, message):
    if not recipients_list:
        return

    print(f"[EMAIL] Envoi de '{subject}' à {len(recipients_list)} personnes...")

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)

        for dest in recipients_list:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_SENDER
            msg['To'] = dest
            msg['Subject'] = subject
            msg.attach(MIMEText(message, 'plain'))
            server.send_message(msg)

        server.quit()
        print("[EMAIL] SUCCÈS.")
    except Exception as e:
        print(f"[EMAIL] ERREUR CRITIQUE : {e}")

def vip_resend_loop(subject, message):
    print(">>> Démarrage du cycle VIP en arrière-plan (Email + Discord)...")
    for i in range(1, 9):
        time.sleep(120)
        print(f">>> Rappel VIP n°{i}/8...")
        send_email_to_list(LISTE_VIP, subject, message)
        send_discord(f"🔔 RAPPEL ({i}/8) : {subject}\n{message}")
    print(">>> Fin cycle VIP.")

def main():
    print("------------------------------------------------")
    print(" DÉMARRAGE DU SCRIPT MARATHON (MODE DISCORD) ")
    print("------------------------------------------------")
    print(f"URL : {URL_TO_CHECK}")
    print(f"VIPs : {LISTE_VIP}")
    print(f"Standards : {LISTE_STANDARD}")
    print("------------------------------------------------")

    # Test de connexion Email
    print("Test de connexion au serveur Email...")
    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)
        server.quit()
        print("-> Connexion Email OK !")
    except Exception as e:
        print(f"-> ERREUR CONNEXION EMAIL : {e}")

    # Envoi de la notification de démarrage (SEULEMENT EMAIL)
    send_email_to_list(LISTE_VIP, "[MARATHON] Script Démarré", "Le script est lancé. Discord sera activé uniquement à l'ouverture.")

    current_hash = get_content_hash(URL_TO_CHECK)

    if current_hash is None:
        print("Impossible de récupérer la page. Arrêt.")
        return

    print("Surveillance active. Attente du prochain cycle...")
    last_heartbeat_date = None

    while True:
        try:
            now = datetime.now()
            current_date = now.date()

            # Heartbeat 9h00 (SEULEMENT EMAIL)
            if now.hour >= 9 and last_heartbeat_date != current_date:
                send_email_to_list(LISTE_VIP, "[MARATHON] Check 9h", "Le script tourne toujours.")
                last_heartbeat_date = current_date

            # Scrapping
            sleep_time = random.uniform(MIN_INTERVAL, MAX_INTERVAL)
            print(f"[{now.strftime('%H:%M:%S')}] Attente de {int(sleep_time)}s avant vérification...")
            time.sleep(sleep_time)

            new_hash = get_content_hash(URL_TO_CHECK)

            if new_hash and new_hash != current_hash:
                print("!!! CHANGEMENT DÉTECTÉ !!!")
                alert_subject = "[MARATHON DU MEDOC] INSCRIPTIONS OUVERTES"
                alert_msg = f"La page d'inscription pour le marathon du médoc est ouverte ! Le lien est : {URL_TO_CHECK}"

                # 1. Email à tout le monde
                all_recipients = LISTE_VIP + LISTE_STANDARD
                send_email_to_list(all_recipients, alert_subject, alert_msg)

                # 2. Discord (ALERT FINALE)
                send_discord(f"🔥🔥 {alert_subject} 🔥🔥\n{alert_msg}")

                # 3. Lancement du spam VIP
                threading.Thread(target=vip_resend_loop, args=(alert_subject, alert_msg)).start()

                current_hash = new_hash
            elif new_hash == current_hash:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] RAS - Aucun changement détecté.")

        except KeyboardInterrupt:
            print("\nArrêt manuel du script.")
            break
        except Exception as e:
            print(f"Erreur dans la boucle principale : {e}")

if __name__ == "__main__":
    main()

------------------------------------------------
 DÉMARRAGE DU SCRIPT MARATHON (MODE DISCORD) 
------------------------------------------------
URL : https://pierre-portfolio.github.io/Pierre/
VIPs : ['petillion99@gmail.com']
Standards : ['quentinlevdso@gmail.com']
------------------------------------------------
Test de connexion au serveur Email...
-> Connexion Email OK !
[EMAIL] Envoi de '[MARATHON] Script Démarré' à 1 personnes...
[EMAIL] SUCCÈS.
Surveillance active. Attente du prochain cycle...
[04:49:45] Attente de 60s avant vérification...
[04:50:46] RAS - Aucun changement détecté.
[04:50:46] Attente de 58s avant vérification...
[04:51:44] RAS - Aucun changement détecté.
[04:51:44] Attente de 59s avant vérification...
[04:52:44] RAS - Aucun changement détecté.
[04:52:44] Attente de 59s avant vérification...
[04:53:43] RAS - Aucun changement détecté.
[04:53:43] Attente de 56s avant vérification...
[04:54:40] RAS - Aucun changement détecté.
[04:54:40] Attente de 57s avant vérifica